- Reload the encoders, scaler and model
- Transform the test case
- Predict

In [1]:
import tensorflow as tf
import pandas as pd
import pickle
from tensorflow.keras.models import load_model

In [2]:
with open('gender_label_encoder.pkl','rb') as f:
    gender_label_encoder=pickle.load(f)

with open('geo_onehot_encoder.pkl','rb') as f:
    geo_onehot_encoder=pickle.load(f)

with open('standard_scaler.pkl','rb') as f:
    standard_scaler=pickle.load(f)

model=load_model('model.keras')


2026-05-10 10:28:47.021572: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-05-10 10:28:47.021872: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-05-10 10:28:47.021892: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-05-10 10:28:47.022440: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-10 10:28:47.022547: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [3]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [4]:
#convert the input dict to df
input_data=pd.DataFrame([input_data])
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [ ]:
#Encode Gender
input_data['Gender']=gender_label_encoder.transform(input_data['Gender'])

In [ ]:
#Encode Geography Column
geo_encoder=geo_onehot_encoder.transform(input_data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1 stored elements and shape (1, 3)>

In [ ]:
#Convert it into a Dataframe
geo_df=pd.DataFrame(geo_encoder.toarray(),columns=geo_onehot_encoder.get_feature_names_out(['Geography']))
geo_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [11]:
#Concat the dataframe
input_data=pd.concat([input_data.drop('Geography',axis=1),geo_df],axis=1)
input_data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [12]:
#Standardize the data
input_data=standard_scaler.transform(input_data)
input_data

array([[-0.51222865,  0.90636285,  0.10591292, -0.69703024, -0.26147196,
         0.8016426 ,  0.64900815,  0.97238125, -0.86754814,  0.99575899,
        -0.57812007, -0.57330877]])

In [13]:
#Predict the churn
prediction=model.predict(input_data)
prediction

2026-05-10 12:15:00.697756: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


array([[0.]], dtype=float32)

In [15]:
prediction_proba = prediction[0][0]

In [16]:
prediction_proba

0.0

In [17]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
